# Creating a DB in sqlite3


## SQL in sqlite3

Using Jupyter magics

In [ ]:
# Capture without a variable sends stdout and stderr to /dev/null
%%capture
%%bash
apt-get update
apt-get install -y sqlite3 wamerican-insane


In [ ]:
!sqlite3 --help
!sqlite3 --version

In [ ]:
!ls -l /usr/share/dict/


In [ ]:
!wc -l /usr/share/dict/american-english-insane


In [ ]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane | wc -l

In [ ]:
%%bash
egrep '^[a-z]{5}$' /usr/share/dict/american-english-insane |
egrep '^...u.$' | wc -l

In [ ]:
%%bash
{
  echo "blah blah blah"
  echo
  echo
  echo words{1..7} | tr ' ' ,
  egrep '^[a-z]{5}$' /usr/share/dict/words |
    shuf -n 420 |
    paste -d, - - - - - - |
    cat -n |
    tr -d ' ' |
    tr '\t' ,
} > /tmp/words.5.csv


In [ ]:
# Show the first few entries
!head /tmp/words.5.csv | cat -nvet


In [ ]:
!ls -la /tmp/words.5.csv


In [ ]:
# Count the number of lines, words, characters
!wc /tmp/words.5.csv


In [ ]:
# Remove any existing database
!rm -rf words.sql3


In [ ]:
# Import CSV into SQLite3
%%script sqlite3 --column --header words.sql3
.mode csv
.import '| grep -A 100000000 ^words /tmp/words.5.csv' words


In [ ]:
# Show that the database now exists
!ls -l


In [ ]:
# Query the table
%%script sqlite3 --column --header words.sql3
select *
from words
limit 10


In [ ]:
# View schema
%%script sqlite3 --column --header words.sql3
.schema


In [ ]:
# Create a table with correct data types, including a primary key ( unique, indexed column )
%%script sqlite3 --column --header words.sql3
CREATE TABLE words_real(
  "words1" INTEGER PRIMARY KEY ASC,
  "words2" TEXT,
  "words3" TEXT,
  "words4" TEXT,
  "words5" TEXT,
  "words6" TEXT,
  "words7" TEXT
);


In [ ]:
# View schema
%%script sqlite3 --column --header words.sql3
.schema


In [ ]:
# Select data into correct table, i.e. "copy" the data
%%script sqlite3 --column --header words.sql3
INSERT INTO words_real
SELECT *
FROM words


In [ ]:
# Show the schema and the first 10 entries
%%script sqlite3 --column --header words.sql3
.schema
select * from words_real limit 10 ;


In [ ]:
# Create an index on the last column
%%script sqlite3 --column --header words.sql3
create index if not exists words7 on words_real (words7) ;


In [ ]:
# Show the schema, noting the index
%%script sqlite3 --column --header words.sql3
.schema


In [ ]:
# Create a derived column
%%script sqlite3 --column --header words.sql3
select *, words6 || "+" || words7 as "both"
from words_real
limit 10


## Using pandas and sqlite3 Python modules

In [ ]:
import sqlite3 as db
import pandas as pd


#### Using a context manager

The context manager automatically closes the connection when finished using the database.


In [ ]:
# Connect to a sqlite database and load a result set into a data frame
with db.connect("words.sql3") as db_con:
  words = pd.read_sql_query("select * from words", db_con)
words


#### Without using a context manager



The connection to the database stays open until you explicitly close it.

In [ ]:
# Connect to a sqlite database and load a result set into a data frame
db_con = db.connect("words.sql3")
words = pd.read_sql_query("select * from words", db_con)
words


In [ ]:
# Show some info about the data frame
words.info()


In [ ]:
# Query the database and load the result into a data frame
words_real = pd.read_sql_query("select * from words_real", db_con)
words_real


In [ ]:
# Show some info about the data frame
words_real.info()


In [ ]:
# Show the first 10 entries in the new data frame
words_real2 = words_real[:10]
words_real2


In [ ]:
# Save the entire dataframe to a new table in the existing database
words_real2.to_sql("words_real2", db_con, if_exists="replace")


In [ ]:
# Show the schema of the entire database
%%script sqlite3 --column --header words.sql3
.schema


Notice that we didn't close the connection.


In [ ]:
# the right thing to do is to close the connection
db_con.close()


In [ ]:
!ls -l
